In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qlinks.models import SquareQDMModel, HoneycombQDMModel, TriangularQDMModel
from qlinks.caging import (
    LocalQDMCageSearcher,
    LocalQDMCageSearchConfig,
    LocalQDMPaddingConfig,
)
from qlinks.caging.classification import (
    classify_cage_state,
    CageClassificationConfig,
)
from qlinks.caging.results import cage_state_to_full_vector
from qlinks.visualizer.basis import BasisGridVisualizer
from qlinks.visualizer import HamiltonianGraphVisualizer, HamiltonianGraphStyle, LinkVisualStyle

## Model definition

In [ ]:
model = SquareQDMModel(
    lx=8,
    ly=8,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    coup_kin=-1.0,
    coup_pot=0.7,
)

## Find caged states candidate from the seed plaquettes with halo

In [ ]:
local_result = LocalQDMCageSearcher.from_plaquettes(
    model,
    plaquette_ids=[0, 2, 4, 6],  # seed plaquettes
    config=LocalQDMCageSearchConfig(
        halo_layers=2,
        boundary_mode="relaxed",
        allowed_kappas=(0,),
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=256,
        ipr_candidate_count=128,
        ipr_random_seed=0,
        tolerance=1e-10,
    ),
).run()

print("local Hilbert size:", local_result.local_hilbert_size)
print("local records:", len(local_result))
print("local counts:", local_result.counts_by_signature)
print("region links:", local_result.region.link_ids.size)
print("active plaquettes:", local_result.region.active_plaquette_ids)
print("unresolved boundary plaquettes:", local_result.region.unresolved_boundary_plaquette_ids)

## Certify the candidates can be embedded into the full lattice

In [ ]:
certified = local_result.certify_paddings(
    config=LocalQDMPaddingConfig(
        max_paddings_per_record=1,
        include_sectors=True,
        require_static_exterior=False,
        tolerance=1e-9,
    )
)

print("certified records:", len(certified))
print("certified counts:", certified.counts_by_signature)
print("limited Hilbert size:", certified.hilbert_size)

In [ ]:
record = certified.records[0]

print("signature:", record.signature)
print("kappa:", record.kappa)
print("potential value:", record.potential_value)
print("support size:", record.cage_state.support_size)
print("energy:", record.cage_state.energy)
print("boundary residual:", record.cage_state.boundary_residual)
print("eigen residual:", record.cage_state.eigen_residual)
print("full residual:", record.cage_state.full_residual)

## Classify the caged state

Important caveat: the classification report is computed on the limited certified Hilbert space. It can not reliably tell whether the reduced-IZ probe is projective or q_empty

In [ ]:
classification_report = classify_cage_state(
    record.cage_state,
    kinetic_matrix=certified.kinetic_matrix,
    basis_configs=certified.basis.states,
    hilbert_size=certified.hilbert_size,
    config=CageClassificationConfig(
        amplitude_tolerance=1e-10,
        cancellation_tolerance=1e-9,
        action_tolerance=1e-9,
        sector_policy="infer_support_component",
    ),
)

print(classification_report)
print("n zeros:", classification_report.n_nontrivial_zeros)

## Visualize the basis states

In [ ]:
grid_visualizer = BasisGridVisualizer(
    lattice=model.lattice,
    layout=model.layout,
    periodic_image_mode="positive_patch",
    collapse_duplicate_visual_links=True,
    style=LinkVisualStyle(
        node_size=60.0,
        site_label_fontsize=4,
        link_label_fontsize=8,
        plaquette_symbol_fontsize=8,
        vulnerable_link_arrow_length_fraction=1.1,
    ),
)

In [ ]:
grid_visualizer.plot_cage_support(
    record,
    basis_configs=certified.basis.states,
    ncols=4,
    show_amplitudes=True,
    amplitude_digits=3,
    show_config_label=False,
    max_states=None,
)

In [ ]:
grid_visualizer.plot_interference_zeros(
    classification_report,
    ncols=4,
    basis_configs=certified.basis.states,
    mechanism="all",
    max_states=None,
)

## Visualize the subgraph

In [ ]:
if record.full_state is not None:
    psi = record.full_state
else:
    psi = cage_state_to_full_vector(record.cage_state, certified.hilbert_size)

print("psi shape:", psi.shape)
print("limited Hilbert size:", certified.hilbert_size)

graph_viz = HamiltonianGraphVisualizer.cage_subgraph_from_sparse_matrix(
    certified.kinetic_matrix,
    state_vector=psi,
    classification_report=classification_report,
    support_tolerance=1e-10,
    include_zero_edges=True,
    include_self_loops=False,
    weight_tolerance=1e-12,
    style=HamiltonianGraphStyle(
        cmap="coolwarm",
        label_vertices=True,
        vertex_size=20,
    ),
)

graph_viz.plot(
    backend="igraph-mpl",
    layout="kk",
    color_by="state_amplitude_real",
    state_vector=graph_viz.graph_data.state_vector,
    title=f"Cage subgraph, signature={record.signature}",
)